In [11]:
import pandas as pd, si_units as si, matplotlib.pyplot as plt, numpy as np, PLOT_SETTINGS as ps, matplotlib.ticker as ticker, ast, os
import feos
from molmass import Formula
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from matplotlib.lines import Line2D
from io import StringIO

plt.rcParams['text.usetex'] = True
plt.rcParams['text.latex.preamble'] = r'\usepackage{xcolor}'


ch4 = Formula("CH4")
n2  = Formula("N2")
co2 = Formula("CO2")
# h2o = Formula("H2O")

molar_masses = np.array([ch4.mass, n2.mass, co2.mass]) # g/mol

def value_in(q, unit):
    """
    Return the numeric value of q in the given unit.
    Works for FeOS/si_units quantities because q/unit -> dimensionless number.
    If q is already numeric, returns q unchanged.
    """
    if isinstance(q, (int, float, np.floating)):
        return float(q)
    return float(q / unit)

def molar_density_mol_m3(rho):
    """
    Convert molar density to mol/m^3.

    Tries, in order:
      - mol/m^3
      - kmol/m^3 (if present) or (1000 mol)/m^3
    """
    mol_per_m3 = si.MOL / (si.METER**3)

    try:
        return value_in(rho, mol_per_m3)
    except Exception:
        pass

    try:
        kmol_per_m3 = si.KMOL / (si.METER**3)
        return value_in(rho, kmol_per_m3) * 1000.0
    except Exception:
        pass

    try:
        kmol_per_m3 = (1000.0 * si.MOL) / (si.METER**3)
        return value_in(rho, kmol_per_m3) * 1000.0
    except Exception as e:
        raise TypeError(f"Could not interpret density units from object {type(rho)}") from e


def density_kg_m3(rho_molar, z, molar_masses, molar_mass_unit="g/mol"):
    """
    rho_molar: FeOS quantity or float (mol/m^3 or kmol/m^3)
    z: mole fractions (dimensionless)
    molar_masses: array of component molar masses (g/mol or kg/mol)
    """
    z = np.asarray(z, dtype=float)
    M = np.asarray(molar_masses, dtype=float)

    if molar_mass_unit.lower() == "g/mol":
        M = M / 1000.0
    elif molar_mass_unit.lower() == "kg/mol":
        pass
    else:
        raise ValueError("molar_mass_unit must be 'g/mol' or 'kg/mol'.")

    rho_mol_m3 = molar_density_mol_m3(rho_molar)   # mol/m^3
    M_mix = np.sum(z * M)                          # kg/mol
    return rho_mol_m3 * M_mix                      # kg/m^3

In [12]:
parameters          = feos.Parameters.from_json(["methane", "nitrogen", "carbon dioxide"], "parameters.json")
pts                 = 500
target_pressure     = 20                         # bar
temperatures        = np.arange(220, 281, 10)    # Kelvin
fluid               = 'Methane&Nitrogen&CO2'              # String, eg. 'Methane&CO2'
# kij                 = kij(temperature)
parameters

|component|molarweight|m|sigma|epsilon_k|k_ij|l_ij|nb|
|-|-|-|-|-|-|-|-|
|methane|16.031|1.0|3.70051|150.07147|0.0|0.0||
|nitrogen|28.006|1.23831|3.30009|89.41358|||2.0|
|carbon dioxide|43.99|2.53096|2.57855|153.31864|0.0|0.0||

In [ ]:
z       = np.array([0.02, 0.02, 0.96])  # mole fractions of CH4, N2, CO2
pcsaft  = feos.EquationOfState.pcsaft(parameters)
T       = 260.0                         # K
P       = 60e5                          # Pa (60 bar)

state = feos.State(pcsaft, temperature=T* si.KELVIN, pressure=P*si.PASCAL, molefracs=z)

In [14]:
rho_from_feos = state.density
rho_kg_m3 = density_kg_m3(rho_from_feos, z, molar_masses, molar_mass_unit="g/mol")
print(rho_kg_m3, "kg/m^3")

972.2078722826094 kg/m^3


In [ ]:
# To compute the other properties, use state.<property_name>, e.g., state.speed_of_sound, heat_capacity_p, etc.

In [29]:
test_prop = state.molar_entropy(feos.Contributions.Residual)
test_prop

-24.312582810247886  J/mol/K